<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [11]</a>'.</span>

# Stage 04c: Feature Encoding + Model Preparation

**Purpose:** Prepare ALL model-ready data

**Inputs:** data/04b_train.parquet, data/04b_test.parquet

**Outputs:**
- data/04c_train_encoded.parquet (after exclusions)
- data/04c_test_encoded.parquet (after exclusions)
- data/04c_train_glm.parquet (raw GLM predictions)
- data/04c_test_glm.parquet (raw GLM predictions)
- config_generated/04c_monotonicity_constraints.yaml

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_pip/v1"


In [3]:
import pandas as pd
import yaml
import os, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "lib"))
from utils import setup_notebook_environment, get_machine_config
from feature_encoder import apply_master_encoding, save_encoders
from model_utils import load_monotonicity_constraints, load_exclusions, load_glm_init, apply_feature_filters

print("########################################")
print("# STAGE 04c: ENCODING + MODEL PREP")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 04c: ENCODING + MODEL PREP
########################################


In [4]:
config_file = f'{config_path}/config.yaml'
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

# Get current machine ID and paths
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'  # current.pc has '3', config key is 'PC3'
output_base = cfg['machines'][pc_id]['paths']['output_path']
aux_data_path = cfg['machines'][pc_id]['paths']['aux_data_path']
print(f'Output: {output_base}')
print(f'Aux data: {aux_data_path}')
vehicle_type = cfg["experiment"]["vehicle_type"]
target = cfg["experiment"]["target"]
exposure = cfg["experiment"]["exposure"]
join_key = cfg["data"]["join_key"]
print(f"Vehicle type: {vehicle_type}")
print(f"Target: {target}")
print(f"Exposure: {exposure}")

Output: output/car_pip/v1
Aux data: /Users/Mach/dev/aps/data/2026_Dmodel_data
Vehicle type: CAR
Target: pp_pip
Exposure: ee_pip_imps


In [5]:
# Load train/test splits
train_file = f'{output_base}/data/04b_train.parquet'
test_file = f'{output_base}/data/04b_test.parquet'

print(f'\n* Loading data...')
train = pd.read_parquet(train_file)
test = pd.read_parquet(test_file)

print(f'  Train: {train.shape}')
print(f'  Test: {test.shape}')


* Loading data...


  Train: (4550028, 121)
  Test: (4539000, 121)


In [6]:
# Apply master encoding
print(f'\n* Applying master feature encoding...')

train_encoded, test_encoded, encoders, encoding_summary = apply_master_encoding(
    train, test, config_path,
    min_frequency=0.01,  # 1% threshold
    min_count=50         # OR 50 observations
)


* Applying master feature encoding...
[OK] Loaded master encoding: 406 columns



[OK] Encoding complete:
  Train shape: (4550028, 199)
  Test shape: (4539000, 199)


In [7]:
# Restore vin_date index (lost during encoding concat)
train_encoded.index = train.index
test_encoded.index = test.index
print(f"  Index restored: {train_encoded.index.name}")

  Index restored: vin_date


In [8]:
# Display encoding summary
print(f'\n* Encoding Summary:')
print(f'\nEncoding types used:')
print(encoding_summary['encoding_type'].value_counts())

print(f'\nCategories with __OTHER__:')
print(encoding_summary[encoding_summary['has_other']== True][['original_column', 'n_categories']].head(10))

print(f'\nCategories with __MISSING__:')
print(encoding_summary[encoding_summary['has_missing'] == True][['original_column', 'n_categories']].head(10))


* Encoding Summary:

Encoding types used:
encoding_type
ordinal_0_5                30
one_hot                    30
binary                     20
numeric                     5
remap_2to4_then_ordinal     5
DROP                        2
skip                        1
Name: count, dtype: int64

Categories with __OTHER__:
                             original_column  n_categories
8                              NumMinAcc_raw             6
9                            NumMajinAcc_raw             6
10                            NumSpdViol_raw             7
11                            NumMinViol_raw            11
12                            NumMajViol_raw             7
24  vc_active_collision_avoidance_system_raw             4
25          vc_active_driving_assistance_raw             4
26          vc_active_parking_assistance_raw             4
27            vc_adaptive_cruise_control_raw             4
29  vc_audible_forward_collision_warning_raw             4

Categories with __MISSING__:


In [9]:
# Apply feature exclusions
print(f"\n* Filtering features...")
exclusions = load_exclusions(config_path, vehicle_type)
all_features = list(train_encoded.columns)
filtered_features = apply_feature_filters(
    all_features, 
    exclusions, 
    target=target, 
    exposure=exposure, 
    join_key=join_key
)
train_encoded = train_encoded[filtered_features]
test_encoded = test_encoded[filtered_features]
print(f"  {len(all_features)} -> {len(filtered_features)} features")


* Filtering features...
Loaded 5 exclusions for CAR
Auto-excluding: target=pp_pip, exposure=ee_pip_imps, join_key=vin_date
Filtered 2 features: ['ee_pip_imps', 'vc_vehicle_immobilizer_raw']
  199 -> 197 features


In [10]:
# Prepare monotonicity constraints
print(f"\n* Preparing monotonicity constraints...")
mono_dict = load_monotonicity_constraints(config_path, vehicle_type, filtered_features)
os.makedirs(f"{output_base}/config_generated", exist_ok=True)
mono_file = f"{output_base}/config_generated/04c_monotonicity_constraints.yaml"
with open(mono_file, "w") as f:
    yaml.dump(mono_dict, f)
print(f"  Saved: {mono_file}")


* Preparing monotonicity constraints...
Loaded 31 monotonicity constraints for CAR
  Saved: output/car_pip/v1/config_generated/04c_monotonicity_constraints.yaml


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [11]:
# GLM initialization
if cfg['model'].get('use_glm_init', False):
    print(f"\n* GLM initialization enabled")
    control_file = cfg['data']['control_model_file']
    target = cfg['experiment']['target'].split('_')[-1]  # e.g., 'coll' from 'incurred_coll'
    target_col = cfg["model"].get("glm_prediction_column", f"pred_{target}")
    
    # Load GLM source file (has unique vin_dates)
    from pathlib import Path
    glm_source_path = Path(aux_data_path) / control_file
    glm_full = pd.read_parquet(glm_source_path, columns=[join_key, target_col])
    
    # Filter to only vin_dates that exist in train/test (keep unique mapping)
    train_vindates = train.index.unique()
    test_vindates = test.index.unique()
    
    glm_train_unique = glm_full[glm_full[join_key].isin(train_vindates)].copy()
    glm_test_unique = glm_full[glm_full[join_key].isin(test_vindates)].copy()
    
    # Rename column to 'glm_pred' for consistency
    glm_train_unique = glm_train_unique.rename(columns={target_col: 'glm_pred'})
    glm_test_unique = glm_test_unique.rename(columns={target_col: 'glm_pred'})
    
    # Save unique vin_date -> glm_pred mapping
    # Set vin_date as index
    glm_train_unique = glm_train_unique.set_index(join_key)
    glm_test_unique = glm_test_unique.set_index(join_key)
    
    glm_train_unique.to_parquet(f"{output_base}/data/04c_train_glm.parquet")
    glm_test_unique.to_parquet(f"{output_base}/data/04c_test_glm.parquet")
    
    print(f"  Saved GLM files:")
    print(f"    Train: {len(glm_train_unique):,} unique vin_dates")
    print(f"    Test: {len(glm_test_unique):,} unique vin_dates")
else:
    print(f"\n* GLM init disabled")


* GLM initialization enabled


ArrowInvalid: No match for FieldRef.Name(pred_pp_pip) in vin_date: string
pred_pp_coll: double
pred_pp_comp: double
pred_pp_liab_1p: double
pred_pp_liab_3p: double
__fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string

In [ ]:
# Save encoded data
train_output = f'{output_base}/data/04c_train_encoded.parquet'
test_output = f'{output_base}/data/04c_test_encoded.parquet'

train_encoded.to_parquet(train_output)
test_encoded.to_parquet(test_output)

print(f'\n* Saved:')
print(f'  {train_output}')
print(f'  {test_output}')

In [ ]:
# Save encoders for holdout
save_encoders(encoders, encoding_summary, output_base)

In [ ]:
print("\n########################################")
print("# STAGE 04c: COMPLETE")
print("########################################")